# Day 10 - 大项目：建自己的预测模型恭喜你来到最后一天！前9天你学到了机器学习的所有基础技能。今天，你要**自己当一回科学家**，从头设计并完成一个天气预测项目。---## 今天的目标1. **自己选择**要预测什么2. **自己设计**特征和模型3. **自己评估**模型效果4. **展示**你的成果

## 1. 选择你的任务机器学习可以解决很多问题。以下是几个你可以选择的方向：| 任务 | 要预测什么 | 难度 ||------|-----------|------|| A. 会不会下雨 | 明天是否下雨（是/否） | ⭐ || B. 下多少雨 | 明天的降雨量（毫米） | ⭐⭐ || C. 明天气温 | 明天的平均温度（度数） | ⭐⭐ || D. 自定义 | 你自己想预测的东西！ | ⭐⭐⭐ |选好了吗？在下面的单元格里写下来你的选择！

In [ ]:
# 在这里写下你选择的任务task = 'A'  # 改成 A, B, C 或 Dprint('你选择的任务是:', task)

## 2. 准备数据无论选择哪个任务，我们都需要先加载和准备数据。

In [ ]:
# 导入库import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import accuracy_score, mean_squared_errorimport warningswarnings.filterwarnings('ignore')plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']plt.rcParams['axes.unicode_minus'] = False# 读取数据df = pd.read_csv('data/weather_sample.csv', encoding='utf-8-sig')df['日期'] = pd.to_datetime(df['日期'])df = df.fillna(df.mean(numeric_only=True))# 日期特征df['月份'] = df['日期'].dt.monthdf['星期'] = df['日期'].dt.dayofweekdf['是否周末'] = (df['星期'] >= 5).astype(int)# 季节特征def get_season(month):    if month in [3, 4, 5]: return 0    elif month in [6, 7, 8]: return 1    elif month in [9, 10, 11]: return 2    else: return 3df['季节'] = df['月份'].apply(get_season)print('数据准备完成！')df.head()

## 3. 自己设计特征这一步**完全由你决定**！想想看：- 你要预测的是什么？- 哪些因素可能影响它？- 你能从现有数据中创造什么新特征？**提示**：- 温度、湿度、气压、风速都是有用的基础特征- 月份和季节可以反映周期性规律- 可以试试把连续值变成类别（如'高/中/低'）- 可以组合多个特征（如温度x湿度）

In [ ]:
# ========== 在这里设计你的特征 ==========feature_cols = [    '温度', '湿度', '风速', '气压',    '月份', '是否周末', '季节']# 如果你想添加自定义特征，可以在这里写：df['温湿指数'] = df['温度'] * df['湿度'] / 100X = df[feature_cols]# 根据你的任务选择标签if task == 'A':    y = df['是否下雨']  # 分类elif task == 'B':    y = df['降雨量']  # 回归elif task == 'C':    y = df['温度']  # 回归else:    y = df['是否下雨']  # 默认分类print('特征数量:', X.shape[1])print('数据量:', X.shape[0])print('标签类型:', y.dtype)

## 4. 训练你的模型现在，选择并训练一个模型！**分类任务**（预测是/否）可以用：- `LogisticRegression` — 逻辑回归- `DecisionTreeClassifier` — 决策树- `RandomForestClassifier` — 随机森林**回归任务**（预测数值）可以用：- `LinearRegression` — 线性回归- `DecisionTreeRegressor` — 决策树回归- `RandomForestRegressor` — 随机森林回归

In [ ]:
# ========== 在这里训练你的模型 ==========X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42)scaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_test_scaled = scaler.transform(X_test)if task == 'A' or task == 'D':    from sklearn.ensemble import RandomForestClassifier    model = RandomForestClassifier(        n_estimators=100, max_depth=5, random_state=42)else:    from sklearn.ensemble import RandomForestRegressor    model = RandomForestRegressor(        n_estimators=100, max_depth=5, random_state=42)model.fit(X_train_scaled, y_train)y_pred = model.predict(X_test_scaled)if task == 'A' or task == 'D':    acc = accuracy_score(y_test, y_pred)    print(f'准确率: {acc*100:.1f}%')else:    rmse = np.sqrt(mean_squared_error(y_test, y_pred))    print(f'均方根误差: {rmse:.2f}')print('模型训练完成！')

## 5. 分析你的结果训练完了！让我们看看模型的表现如何。

In [ ]:
# 查看特征重要性importances = model.feature_importances_feat_imp = pd.DataFrame({'特征': feature_cols, '重要性': importances})feat_imp = feat_imp.sort_values('重要性', ascending=True)plt.figure(figsize=(8, 5))plt.barh(feat_imp['特征'], feat_imp['重要性'], color='coral')plt.xlabel('重要性', fontsize=14)plt.title('你的模型 - 特征重要性', fontsize=16)plt.tight_layout()plt.show()print('对你模型最重要的三个特征:')print(feat_imp.tail(3).to_string(index=False))

In [ ]:
# 可视化预测效果if task == 'A' or task == 'D':    from sklearn.metrics import confusion_matrix    cm = confusion_matrix(y_test, y_pred)    plt.figure(figsize=(5, 4))    plt.imshow(cm, cmap='Oranges')    plt.title('混淆矩阵', fontsize=16)    plt.xlabel('预测', fontsize=12)    plt.ylabel('真实', fontsize=12)    plt.xticks([0, 1], ['否', '是'])    plt.yticks([0, 1], ['否', '是'])    for i in range(2):        for j in range(2):            plt.text(j, i, str(cm[i][j]), ha='center', va='center', fontsize=18)    plt.colorbar()    plt.tight_layout()plt.show()else:    plt.figure(figsize=(6, 6))    plt.scatter(y_test, y_pred, alpha=0.3, color='teal')    min_val = min(y_test.min(), y_pred.min())    max_val = max(y_test.max(), y_pred.max())    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)    plt.xlabel('真实值', fontsize=14)    plt.ylabel('预测值', fontsize=14)    plt.title('预测 vs 真实', fontsize=16)    plt.tight_layout()plt.show()

## 6. 记录你的实验科学家做实验时都会记录。请在下面写下你的发现：

In [ ]:
# ========== 实验记录 ==========# 在这里写下你的发现：# 1. 我选择的任务是：___# 2. 我使用的特征有：___# 3. 我选择的模型是：___# 4. 最终的效果是：准确率 ___ / 误差 ___# 5. 最重要的特征是：___# 6. 我尝试过的改进有：___# 7. 我的结论是：___print('=' * 40)print('       我的实验报告')print('=' * 40)print('任务: 预测明天是否下雨')print('特征:', feature_cols)print('模型: RandomForest')print('最重要的特征:', feat_imp.iloc[-1]['特征'])print('=' * 40)

---## 恭喜你完成了10天的机器学习之旅！回顾一下你学到的所有技能：| 天数 | 技能 ||------|------|| Day 01 | Python 编程基础 || Day 02 | 数据可视化 || Day 03 | 线性回归 || Day 04 | 分类问题 || Day 05 | 决策树 || Day 06 | 模型评估 || Day 07 | 聚类分析 || Day 08 | 特征工程 || Day 09 | 完整项目实战 || Day 10 | 独立设计模型 |---## 接下来可以做什么？1. **参加 Kaggle 比赛** — 用真实数据挑战自己2. **学习深度学习** — 用神经网络做更复杂的预测3. **做更多项目** — 预测股票、分析文本、识别图像...4. **教别人** — 最好的学习方式是教别人！**机器学习的世界很大，你才刚刚开始。继续探索吧！**